In [3]:
# ====================== 环境准备 ======================
# 1) 设定环境变量（必须在导入 matplotlib 之前执行）
import os
from copy import deepcopy
from copy import deepcopy

os.environ['QT_API'] = 'pyqt5'        # 指定使用 PyQt5 作为 Qt 绑定
os.environ['MPLBACKEND'] = 'QtAgg'    # 指定 Matplotlib 后端为 QtAgg（更推荐，替代 TkAgg）

# 2) 启动 Qt 事件循环（控制台模式下，让 Qt 窗口能实时响应）
%gui qt5
from copy import deepcopy

# 3) 检查 matplotlib backend
import matplotlib as mpl
mpl.rcParams.update({
    # 这里按系统常见字体给一串候选，存在则自动生效
    "font.sans-serif": ["Microsoft YaHei", "SimHei", "SimSun",
                        "Noto Sans CJK SC", "Source Han Sans SC",
                        "Arial Unicode MS", "DejaVu Sans"],
    "font.family": "sans-serif",
    "axes.unicode_minus": False,   # 负号用正常字符，避免被当作缺字形
})

print("backend (before pyplot):", mpl.get_backend())
# 如果不是 QtAgg，强制改为 QtAgg（注意：必须在导入 pyplot 前设置）
mpl.rcParams['backend'] = 'QtAgg'

# 4) 现在再导入 pyplot
import matplotlib.pyplot as plt
print("backend (after pyplot):", mpl.get_backend())

# ====================== 导入依赖 ======================
import sys
# 避免反复执行时 Qt 类重复导入导致崩溃：如果已加载，先删除再导入
if 'draw.pyqt_draw.pyqt_main2' in sys.modules:
    del sys.modules['draw.pyqt_draw.pyqt_main2']

from PyQt5 import QtWidgets
import pyqtgraph as pg
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
import draw.read_snap_xml  as read_snap_xml

# 配置 pyqtgraph：开启抗锯齿，关闭 OpenGL（更稳定）
pg.setConfigOptions(antialias=True)
# pg.setConfigOptions(useOpenGL=False)   # 若驱动或 OpenGL 有问题可显式关闭
# ====================== 基础参数 ======================
# 星座参数：每轨道卫星数 N，轨道平面数 P
N = 36
P = 18

from config import DATA_DIR,INPUT_DIR
from pathlib import Path

backend (before pyplot): qtagg
backend (after pyplot): QtAgg


In [4]:
from draw.pyqt_draw.pyqt_main2 import SatelliteViewer
SIMULATION_EDITION = 'motif2'

In [10]:
time_2_build = 60
TIME_2_BUILD = time_2_build

# 默认用 "topology_{TIME_2_BUILD}"，也允许用环境变量 TOPOLOGY_VERSION 覆盖
DEFAULT_VERSION = f"{SIMULATION_EDITION}/topology_{TIME_2_BUILD}"
VERSION = os.getenv("TOPOLOGY_VERSION", DEFAULT_VERSION)


RAW_DIR    = Path(INPUT_DIR) / VERSION / "raw"

CONFIG_DIR = Path(INPUT_DIR) / f"{SIMULATION_EDITION}/config"
MODIFY_DIR    = Path(INPUT_DIR) / VERSION / "modify"
# 若不存在则创建（递归创建上级目录；已存在不报错）
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
import genaric2.tegnode as tegnode
import draw.basic_functio.write2xml as write2xml

# 这里，我们读取到nodes 信息,但是我们需要转化为edge信息，注意这里主要就只有包括inter-edge信息


True


In [7]:
file_in = DATA_DIR
# xml_file = r"DATA_DIR\station_visible_satellites_648_1d_real.xml"
xml_file = DATA_DIR / "station_visible_satellites_648_1d_real.xml"

def slice_group_data(raw_group_data, start, end):
    """
    从 raw_group_data 中裁剪时间区间 [basicSa, end)
    """
    return {
        step: raw_group_data[step]
        for step in range(start, end)
        if step in raw_group_data
    }
# 只做一次：解析大区间
RAW_START, RAW_END = 0, 22006
raw_group_data = read_snap_xml.parse_xml_group_data(xml_file, RAW_START, RAW_END)


In [8]:
import importlib
importlib.reload(write2xml)

True


<module 'draw.basic_functio.write2xml' from 'C:\\usrspace\\mywork\\generic\\draw\\basic_functio\\write2xml.py'>

In [12]:
# RANGES = [
#     (0,1204),(1204,3669),(3669,4094),(4094,6814),(6814,8485),(8485,11640),
#      (11640,13057),(13057,14065),(14065,16604),(16604,18396),
#     (18396,19831),(19831,20814),(20814,22005)
# ]
RANGES = [
    (0,1204),(1204,3669),(3669,4094),(4094,6814)
]


paths = [MODIFY_DIR /f"interplane_links_{s}_{e}.xml" for s, e in RANGES]

# 方式 A：顺序（内存最低、稳定）
totalnode = write2xml.load_all_nodes_sequential_test(paths, tegnode.tegnode_new)

# 方式 B：并行（SSD + lxml 时建议用；workers 可按盘/CPU调整）这个很慢
# totalnode = write2xml.load_all_nodes_parallel(paths, tegnode.tegnode_complete,
#                                     workers=12, backend='thread')


In [13]:
start_ts =0
end_ts = 6814

In [14]:
group_data = slice_group_data(raw_group_data, start_ts, end_ts)


In [17]:
rev_group_data,offset = read_snap_xml.modify_group_data(group_data,P=18, N=36, base_groupid=4)



In [18]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
# 注意 ，下面是直接将nodes转为edge，因为我们的nodes本身已经完成了冲突检测和处理
# import draw.basic_functio.inter_edge2nodes as inter_edge2nodes
# all_inter_edge = inter_edge2nodes.trans_nodes2edges(totalnode,P,N)

import draw.basic_functio.motif as motif
edges_by_step,pending_edge = motif.transform_nodes_2_rawedge_test(totalnode, P, N, start_ts, end_ts)


In [23]:
from typing import Dict, Set, Tuple, Iterable
import pandas as pd

Adj = Dict[int, Set[int]]                 # 邻接：u -> {v1, v2, ...}
AllAdj = Dict[int, Adj]                   # 时刻 -> 邻接

def _edges_from_adj(adj: Adj, directed: bool=False) -> Set[Tuple[int,int]]:
    """
    把邻接字典转成边集合。
    - 无向：边一律存成 (min(u,v), max(u,v))，避免重复计数
    - 有向：边存成 (u, v)
    """
    edges: Set[Tuple[int,int]] = set()
    for u, nbrs in adj.items():
        if not isinstance(nbrs, Iterable):
            continue
        for v in nbrs:
            if u == v:
                continue
            if directed:
                edges.add((u, v))
            else:
                a, b = (u, v) if u < v else (v, u)
                edges.add((a, b))
    return edges

def summarize_topology(all_inter_edge: AllAdj, directed: bool=False) -> pd.DataFrame:
    """
    统计每时刻的链路数量与拓扑是否变化（相对前一时刻）。
    返回列：
      - t: 时刻
      - link_count: 边条数
      - changed: 是否变化（0/1）
      - add_edges: 本时刻相对上一时刻 新增的边数量
      - del_edges: 本时刻相对上一时刻 删除的边数量
    """
    times = sorted(all_inter_edge.keys())
    rows = []
    prev_edges: Set[Tuple[int,int]] = set()

    for i, t in enumerate(times):
        edges_t = _edges_from_adj(all_inter_edge[t], directed=directed)
        link_count = len(edges_t)

        if i == 0:
            changed = 0
            add_cnt = del_cnt = 0
        else:
            add = edges_t - prev_edges
            rem = prev_edges - edges_t
            add_cnt, del_cnt = len(add), len(rem)
            changed = 1 if (add_cnt or del_cnt) else 0

        rows.append({
            "t": t,
            "link_count": link_count,
            "changed": changed,      # 0 = 无变化，1 = 有变化
            "add_edges": add_cnt,
            "del_edges": del_cnt,
        })
        prev_edges = edges_t

    df = pd.DataFrame(rows)
    return df

# ===== 示例：用你的 all_inter_edge 运行 =====
df = summarize_topology(all_inter_edge, directed=False)
# 查看：print(df.head())
# 想导出 Origin/CSV：df.to_csv("topology_change_summary.csv", index=False)
# df.to_csv("topology_change_summary.csv", index=False)

In [24]:
import draw.pymatlab2.chartalgorithm.export_topology_summary as export_topology_summary
paths = export_topology_summary.export_topology_summary(
    all_inter_edge,
    out_dir=FIGURE_DIR,
    basename=f"topology_snapshot_ttb{TIME_2_BUILD}",
    to=("csv"),      # 想要什么格式就写什么
    directed=False,
    with_helpers=True,       # 生成 stable_line / change_line / change_mark
    with_alt_group=True      # 生成 color_group（0/1 交替，用于分段上色）
)
print(paths)


{'csv': WindowsPath('C:/usrspace/mywork/generic/data/input/topology_60/figure/topology_snapshot_ttb60.csv')}


In [ ]:
df

In [25]:

# 这里，我们需要将nodes 信息转化为edge信息,注意，这个edge也只有inter-edge，并且是原始
# 这里，我们还需要将pending edges 信息也加入到all_inter_edge 中
pending_edges = inter_edge2nodes.trans_nodes2_pendingedges2(totalnode, start_ts, end_ts, time_2_build, P, N)

In [19]:
# 转化为inter-edge信息后，我们可以通过绘图来初步查看

# ====================== 绘图初始化 ======================
# 1) QApplication 实例（全局唯一）
app = QtWidgets.QApplication.instance() or QtWidgets.QApplication([])

# 2) 确保 viewer 有全局引用，避免 GC 回收导致崩溃
if not hasattr(sys.modules[__name__], "_viewer_list"):
    _viewer_list = []


In [21]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("raw behand")
viewer.resize(1200, 700)

viewer.edges_by_step = edges_by_step
viewer.pending_links_by_step=   pending_edge
viewer.show()



In [ ]:
# 3) 创建并配置 viewer
viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("raw behand")
viewer.resize(1200, 700)

viewer.show


注意，上述的拓扑，是只有异轨链路的，并且，在邻接表上，也是单向的。因此，我们实际上要做这几件事情

1.实际网络拓扑是还有同轨链路的，所以，我们还是要加上同轨链路信息

2.在邻接表上，我们需要将所有的边都转化为双向的

In [ ]:
#添加同轨链路
# all_intra_edge = {}
# for step in range(start_ts, end_ts):
#     all_intra_edge[step] = {}
#     for i in range(P):
#         for j in range(N):
#             nownode = i * N + j
#             nextnode = i * N + (j + 1) % N
#             upnode = i * N + (j - 1 + N) % N
#             all_intra_edge[step].setdefault(nownode, set()).add(nextnode)
#             all_intra_edge[step].setdefault(nownode, set()).add(upnode)


def build_intra_edges_copies(start_ts, end_ts, P, N):
    # 预计算每个节点的左右邻居（tuple 轻量不可变，便于快速构造 set）
    base_neighbors = {
        i * N + j: (i * N + ((j + 1) % N), i * N + ((j - 1) % N))
        for i in range(P) for j in range(N)
    }

    all_intra_edge = {}
    for step in range(start_ts, end_ts):
        # 一次性构造（避免 setdefault & 多次 add 的开销）
        adj = {node: set(neis) for node, neis in base_neighbors.items()}
        all_intra_edge[step] = adj
    return all_intra_edge

# 用法
all_intra_edge = build_intra_edges_copies(start_ts, end_ts, P, N)


In [ ]:
def make_edges_bidirectional(edge_dict):
    """
    edge_dict: {basicSa: set([dst, ...]), ...}
    返回双向邻接表
    """
    new_edges = {}
    for src, dsts in edge_dict.items():
        for dst in dsts:
            new_edges.setdefault(src, set()).add(dst)
            new_edges.setdefault(dst, set()).add(src)
    return new_edges
for step in all_inter_edge:
    all_inter_edge[step] = make_edges_bidirectional(all_inter_edge[step])


In [ ]:
all_edges = {}

for step in range(start_ts, end_ts):
    all_edges[step] = {}
    # 先合并intra_edge
    if step in all_intra_edge:
        for src, dsts in all_intra_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)
    # 再合并inter_edge
    if step in all_inter_edge:
        for src, dsts in all_inter_edge[step].items():
            all_edges[step].setdefault(src, set()).update(dsts)


In [ ]:

viewer = SatelliteViewer(group_data)
viewer.setWindowTitle("prove the nodes information")
viewer.resize(1200, 700)
viewer.edges_by_step = all_edges
viewer.pending_links_by_step = pending_edges
viewer.show()
_viewer_list.append(viewer)

In [ ]:
import  importlib
importlib.reload(plot_intergroup_avg_shortest_path)

接下来，我们可以查看在这段时间内，平均最短路径的变化情况


In [ ]:
# 1) 只在 Jupyter 里非阻塞显示
import  draw.pymatlab2.chartalgorithm.plot_intergroup_avg_shortest_path as plot_intergroup_avg_shortest_path
# plot_intergroup_avg_shortest_path.plot_intergroup_avg_shortest_path(all_edges, group_data, group_a=0, group_b=4)

# 假设已有 all_edges, group_data
csv_path = plot_intergroup_avg_shortest_path.export_intergroup_avgspath_to_origin(
    all_edges, group_data,
    out_dir=FIGURE_DIR,
    basename=f"avgspath_g0_4_ttb{TIME_2_BUILD}",
    group_a=0, group_b=4,
    steps=(0, 22005),         # 或 None / 自定义迭代器
    undirected=True
)
print("written:", csv_path)


# # 2) 显示 + 保存 PNG/PDF
# plot_intergroup_avg_shortest_path(
#     all_edges, group_data,
#     group_a=0, group_b=4,
#     save=True,                 # 用默认保存规则
#     save_dir=FIGURE_DIR,       # 你的 figure 目录
#     basename="avgspath_g0_g4",
#     formats=("png","pdf"),
#     dpi=300
# )
#
# # 3) 不显示只保存，拿到 DataFrame 做后续处理
# fig, ax, df_metric = plot_intergroup_avg_shortest_path.plot_intergroup_avg_shortest_path(
#     all_edges, group_data,
#     save_dir=FIGURE_DIR,
#     show=False,
#     save=["avgspath.png", "avgspath.pdf"]
# )


In [ ]:
FIGURE_DIR

In [ ]:
import  importlib
importlib.reload(plot_switches_nonblocking)

1. 写成一个专门的函数 2. 在jupter里运行的时候，能够非阻塞 3.我能够选择是否保存图片和pdf

In [ ]:
##这个代码是显示,每时刻正在建链的链路条数
# 目前已经已经实现
#1 . 直接运行显示
#2.数据输出csv输出交由origin处理


# 平均切换条数显示
import  draw.pymatlab2.chartalgorithm.plot_switches_nonblocking as plot_switches_nonblocking
# 生成切换图
# 1) 只在 Jupyter 里“非阻塞显示”，不保存
# plot_switches_nonblocking.plot_pending_edges_timeseries(pending_edges)

# 2) 显示 + 同时保存 PNG 和 PDF（论文友好）

# plot_switches_nonblocking.plot_pending_edges_timeseries(
#     pending_edges,
#     save=True,
#     save_dir=FIGURE_DIR,
#     basename="pending_30s_4k",
#     formats=("png","svg","pdf"),
#     dpi=300,
#     save_pixels=(3840, 2160),   # ★ 4K
#     keep_display_size=True
# )


# # 3) 不显示、只保存到指定路径（多格式）
# plot_pending_edges_timeseries(
#     pending_edges,
#     show=False,
#     save=["out/pending_edges.png", "out/pending_edges.pdf"]
# )
#
# # 4) 获取返回的 DataFrame，后续自定义处理
# fig, ax, df_counts = plot_pending_edges_timeseries(pending_edges, return_handles=True)
# df_counts.head()
#

#导出
# 你已有：FIGURE_DIR = Path(INPUT_DIR) / VERSION / "figure"
# 没有就先建


# 导出 CSV（Origin 里：数据 → 从文件导入 → 单一 ASCII）
paths = plot_switches_nonblocking.export_pending_series_to_origin(
    pending_edges,
    out_dir=FIGURE_DIR,
    basename="pending_edges_ttb60",
    to=("csv",),            # 也可 ("csv","xlsx")
    fill_missing=True,
    max_t_seconds=None,
    smooth_window=None         # 想要多一列移动平均就填窗口大小；不需要就设 None
)
print(paths)


In [ ]:
FIGURE_DIR

下面是计算平均切换条数

In [ ]:
def average_switches(pending_edges):
    total_edges = 0
    total_steps = 0

    for step, edge_dict in pending_edges.items():
        # edge_dict: dict[src_id] -> set(dst_ids)
        step_edges = sum(len(dsts) for dsts in edge_dict.values())
        total_edges += step_edges
        total_steps += 1

    if total_steps == 0:
        return 0.0

    return total_edges / total_steps

avg = average_switches(pending_edges)
print(f"平均切换跳数: {avg:.2f}")


In [ ]:
# 计算平均切换数（全范围）
import  draw.pymatlab2.chartalgorithm.plot_pending_edges_timeseries as plot_pending_edges_timeseries
avg = plot_pending_edges_timeseries.average_switches(pending_edges)
print(f"平均切换数: {avg:.2f}")

# 在 Jupyter 非阻塞显示，并标注平均值；同时保存 PNG/PDF 到 FIGURE_DIR
fig, ax, df_counts, avg2 = plot_pending_edges_timeseries.plot_pending_edges_timeseries(
    pending_edges,
    max_t_seconds=None,       # 或 22000
    annotate_avg=True,
    save=True,
    save_dir=FIGURE_DIR,      # 你之前定义的目录
    basename="pending_30s",
    formats=("png","pdf"),
    dpi=300
)


下面将

链路切换与平均最短路径画在一起，这样能够看出链路切换对于平均最短路径的影响


In [ ]:
# import math
# import networkx as nx
# import numpy as np
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FormatStrFormatter
#
# # ===== 可调参数 =====
# MAX_T_SECONDS = None   # 例如 5000；None 表示全程
# FILL_MISSING  = True   # 缺失的秒是否填 0（让切换曲线连续）
# LINE_WIDTH    = 2.0
#
# # ====== 计算函数（平均跳数）======
# def compute_avg_hops_per_step(all_edges, group_data, steps_sorted):
#     """
#     返回与 steps_sorted 对齐的 avg_path_lengths 列表（无路用 NaN）
#     all_edges: {t: {basicSa: set(dst)}}
#     group_data: {t: {'groups': {0:set, 4:set}, ...}}
#     """
#     avg_path_lengths = []
#
#     for step in steps_sorted:
#         # group 合法性检查
#         gd = group_data.get(step)
#         if not gd or 0 not in gd['groups'] or 4 not in gd['groups']:
#             avg_path_lengths.append(float('nan'))
#             continue
#
#         group0 = gd['groups'][0]
#         group4 = gd['groups'][4]
#         if not group0 or not group4:
#             avg_path_lengths.append(float('nan'))
#             continue
#
#         # 构造无向图（一次性列出边，避免 add_edge 循环的 Python 开销）
#         mapping = all_edges.get(step, {})
#         if not mapping:
#             avg_path_lengths.append(float('nan'))
#             continue
#
#         edges = [(s, d) for s, dsts in mapping.items() for d in dsts]
#         G = nx.Graph()
#         G.add_edges_from(edges)
#
#         # 只对 group0 的每个点跑一次 BFS，提取到 group4 的长度
#         g4_set = set(group4)
#         pair_sum = 0
#         pair_cnt = 0
#         for n0 in group0:
#             if n0 not in G:  # 这个源点没边
#                 continue
#             lengths = nx.single_source_shortest_path_length(G, n0)
#             # 只取目标在 group4 的
#             for n4 in g4_set:
#                 if n4 in lengths:
#                     pair_sum += lengths[n4]
#                     pair_cnt += 1
#
#         avg = (pair_sum / pair_cnt) if pair_cnt > 0 else float('nan')
#         avg_path_lengths.append(avg)
#
#     return avg_path_lengths
#
# # ====== 统计每秒切换边数 ======
# def compute_edge_counts(pending_edges, steps_sorted, fill_missing=True):
#     """
#     统计每一秒的边数，与 steps_sorted 对齐。
#     fill_missing=True 时，对 pending_edges 缺失的秒计为 0
#     """
#     def count_edges_at_t(t):
#         ed = pending_edges.get(t, {})
#         return sum(len(dsts) for dsts in ed.values())
#
#     if fill_missing:
#         # steps_sorted 已是连续时间轴的话直接用；否则外层会统一生成连续轴
#         return [count_edges_at_t(t) for t in steps_sorted]
#     else:
#         # 仅保留 pending_edges 里存在的秒（通常不建议与另一条曲线合并时使用）
#         return [count_edges_at_t(t) if t in pending_edges else np.nan for t in steps_sorted]
#
# # ====== 主流程：统一时间轴，计算两条曲线并合并绘图 ======
# # 统一时间轴：用两侧数据的并集，并（可选）补齐为连续秒
# steps_edges = set(all_edges.keys())
# steps_pends = set(pending_edges.keys())
# if group_data:
#     steps_groups = set(group_data.keys())
# else:
#     steps_groups = set()
#
# # 时间轴取并集（更稳妥），再按需变为连续
# steps_union = sorted(steps_edges | steps_pends | steps_groups)
# if FILL_MISSING and steps_union:
#     t_min, t_max = steps_union[0], steps_union[-1]
#     steps = list(range(t_min, t_max + 1))
# else:
#     steps = steps_union
#
# # 计算平均跳数（如果你之前已经有 avg_path_lengths/steps 对，就可以跳过这一步）
# avg_path_lengths = compute_avg_hops_per_step(all_edges, group_data, steps_sorted=steps)
#
# # 计算切换边数
# edge_counts = compute_edge_counts(pending_edges, steps_sorted=steps, fill_missing=True)
# import matplotlib as mpl
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FormatStrFormatter
#
# def plot_latency_vs_switches(
#     steps, avg_path_lengths, edge_counts, max_t=None,
#     color_latency="tab:blue", color_switch="tab:orange"
# ):
#     """
#     上下两个子图对齐：
#       上：平均最短路径跳数（延迟）
#       下：每秒切换边数（切换频率）
#     """
#     # 可选裁剪
#     if max_t is not None and steps:
#         mask = [t <= max_t for t in steps]
#         steps = [t for t, m in zip(steps, mask) if m]
#         avg_path_lengths = [v for v, m in zip(avg_path_lengths, mask) if m]
#         edge_counts      = [v for v, m in zip(edge_counts,      mask) if m]
#
#     rc = {
#         "font.family": "Times New Roman",
#         "font.size": 14,
#         "axes.labelsize": 18,
#         "axes.titlesize": 18,
#         "xtick.labelsize": 12,
#         "ytick.labelsize": 12,
#         "axes.linewidth": 1.2,
#     }
#     with mpl.rc_context(rc):
#         fig, (ax_top, ax_bot) = plt.subplots(
#             2, 1, sharex=True, figsize=(11.5, 6.5),
#             gridspec_kw={"height_ratios": [2.2, 1.3], "hspace": 0.06}
#         )
#
#         # —— 上：平均跳数（延迟）
#         ax_top.plot(steps, avg_path_lengths, lw=2.0, color=color_latency, label="Avg hops (Group0↔Group4)")
#         ax_top.set_ylabel("Average shortest-path hops", color=color_latency)
#         ax_top.tick_params(axis='y', colors=color_latency)
#         ax_top.grid(True, linestyle="--", alpha=0.35)
#
#         # —— 下：切换边数（用阶梯线更易读；也可改成 bar）
#         ax_bot.plot(steps, edge_counts, lw=2.0, color=color_switch,
#                     drawstyle="steps-mid", label="Edges per second")
#         ax_bot.set_ylabel("Edges per second", color=color_switch)
#         ax_bot.tick_params(axis='y', colors=color_switch)
#         ax_bot.ticklabel_format(style="plain", axis="y", useOffset=False, useMathText=False)
#         ax_bot.yaxis.set_major_formatter(FormatStrFormatter('%d'))
#         ax_bot.set_xlabel("Time (s)")
#         ax_bot.grid(True, linestyle="--", alpha=0.30)
#
#         # 统一图例（放在右侧外部）
#         lines = [
#             ax_top.get_lines()[0],
#             ax_bot.get_lines()[0],
#         ]
#         labels = [l.get_label() for l in lines]
#         plt.tight_layout(rect=[0, 0, 0.82, 1.0])  # 右侧留白
#         leg = fig.legend(lines, labels, loc="center left",
#                          bbox_to_anchor=(0.86, 0.5), frameon=True, framealpha=1.0)
#         leg.get_frame().set_edgecolor("black")
#         plt.show()
#
# # —— 如果你已经有 steps / avg_path_lengths / edge_counts：
# plot_latency_vs_switches(steps, avg_path_lengths, edge_counts, max_t=22000)
#
#


这里是用来查看切换的周期的

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ---- 可选：如果你还没有 edge_counts / steps，可以用这段从 pending_edges 统计 ----
def compute_edge_counts_from_pending(pending_edges, fill_missing=True):
    all_steps = sorted(pending_edges.keys())
    tmin, tmax = min(all_steps), max(all_steps)
    if fill_missing:
        steps = np.arange(tmin, tmax + 1, dtype=int)
        counts = np.zeros_like(steps, dtype=int)
        for t, i in zip(steps, range(len(steps))):
            ed = pending_edges.get(int(t), {})
            counts[i] = sum(len(s) for s in ed.values())
    else:
        steps = np.array(all_steps, dtype=int)
        counts = np.array([sum(len(s) for s in pending_edges[t].values()) for t in steps], dtype=int)
    return steps, counts

# steps, edge_counts = compute_edge_counts_from_pending(pending_edges, fill_missing=True)

# =================== 1) 滑动自相关取局部主周期 ===================
def rolling_acf_period(counts, steps,
                       win_sec=600, hop_sec=30,
                       period_min=30, period_max=1800,
                       acf_peak_min=0.1):
    """
    counts: 每秒切换数（与 steps 对齐，等间隔）
    win_sec: ACF 窗口长度（秒）
    hop_sec: 窗口滑动步长（秒）
    period_min/max: 只在该区间内寻找周期（秒）
    acf_peak_min: 认为是“显著峰”的最小自相关值
    返回：times_center, local_period (秒；无解处为 NaN)
    """
    x = counts.astype(float)
    N = len(x)
    fs = 1.0  # 秒采样
    win = int(win_sec)
    hop = int(hop_sec)
    if N < win or win < 8:
        return np.array([]), np.array([])

    # 去趋势（每窗减均值），效果更稳
    def acf_via_fft(sig):
        # 零均值
        sig = sig - sig.mean()
        if np.allclose(sig, 0):
            return np.zeros_like(sig)
        nfft = 1
        L = len(sig)
        while nfft < 2*L:
            nfft <<= 1
        S = np.fft.rfft(sig, n=nfft)
        acf = np.fft.irfft(np.abs(S)**2, n=nfft)[:L]
        # 归一化（除以长度 & 方差）
        acf = acf / (np.arange(L, 0, -1))  # 无偏
        v = sig.var()
        if v > 0:
            acf = acf / v
        return acf

    t_list, p_list = [], []
    i = 0
    while i + win <= N:
        seg = x[i:i+win]
        acf = acf_via_fft(seg)
        # 在 [min,max] 对应的滞后范围寻找“第一个显著峰”
        lag_min = int(np.ceil(period_min*fs))
        lag_max = int(min(len(acf)-1, np.floor(period_max*fs)))
        if lag_max > lag_min:
            y = acf[lag_min:lag_max+1]
            # 简单且快的局部极大值检测
            # maxima at j if y[j] > y[j-1] and y[j] >= y[j+1]
            if len(y) >= 3:
                left  = y[1:-1] > y[:-2]
                right = y[1:-1] >= y[2:]
                peaks = np.where(left & right)[0] + 1
            else:
                peaks = np.array([], dtype=int)

            # 在显著阈值上选“第一个”峰
            period_val = np.nan
            for pk in peaks:
                if y[pk] >= acf_peak_min:
                    lag = lag_min + pk
                    period_val = lag / fs
                    break
        else:
            period_val = np.nan

        t_center = steps[i + win//2]
        t_list.append(t_center)
        p_list.append(period_val)
        i += hop

    return np.array(t_list), np.array(p_list)

# =================== 2) STFT 频谱→周期“热力图” ===================
def stft_spectrogram(counts, steps,
                     win_sec=600, hop_sec=30,
                     period_min=30, period_max=1800):
    """
    返回：times_center, periods(纵轴), Pxx(功率谱密度)
    """
    fs = 1.0
    x = counts.astype(float)
    N = len(x)
    win = int(win_sec)
    hop = int(hop_sec)
    if N < win:
        return np.array([]), np.array([]), np.empty((0,0))

    # 汉宁窗
    w = np.hanning(win)
    nfft = 1
    while nfft < win:
        nfft <<= 1

    t_list, spec_list = [], []
    i = 0
    while i + win <= N:
        seg = x[i:i+win] * w
        S = np.fft.rfft(seg, n=nfft)
        Pxx = (np.abs(S)**2) / (np.sum(w**2))
        t_list.append(steps[i + win//2])
        spec_list.append(Pxx)
        i += hop

    T = np.array(t_list)
    Sxx = np.vstack(spec_list)  # shape (frames, freqs)

    freqs = np.fft.rfftfreq(nfft, d=1/fs)  # Hz
    # 频率→周期（秒），去掉 0 频
    f = freqs[1:]
    periods = 1.0 / f
    Sxx = Sxx[:, 1:]

    # 只保留 [period_min, period_max]
    mask = (periods >= period_min) & (periods <= period_max)
    return T, periods[mask], Sxx[:, mask]

# =================== 3) 峰间间隔（及滚动中位数） ===================
def peak_intervals(counts, min_distance=20, smooth_win=31, thresh_rel=0.2):
    """
    简单峰检测（无 SciPy 版本）：
      - 滑动平均平滑
      - 选高于 (min + thresh_rel*(max-min)) 的点作为候选
      - 取相邻上升-下降构成峰，保持最小间隔
    返回：peak_indices, intervals(秒)
    """
    x = counts.astype(float)
    if smooth_win > 1:
        k = smooth_win
        pad = k//2
        x_pad = np.pad(x, (pad,pad), mode='edge')
        kernel = np.ones(k)/k
        xs = np.convolve(x_pad, kernel, mode='valid')
    else:
        xs = x

    # 自适应阈值
    lo, hi = xs.min(), xs.max()
    thr = lo + thresh_rel*(hi-lo)

    # 简易峰：从低到高再到低
    peaks = []
    last_idx = -np.inf
    rising = False
    for i in range(1, len(xs)-1):
        if xs[i] >= thr:
            if xs[i] > xs[i-1] and xs[i] >= xs[i+1]:
                if (i - last_idx) >= min_distance:
                    peaks.append(i)
                    last_idx = i
    peaks = np.array(peaks, dtype=int)
    intervals = np.diff(peaks)  # 秒
    return peaks, intervals

# =================== 4) 统一画图 ===================
def plot_switching_period_diagnostics(steps, edge_counts,
                                      acf_win=600, acf_hop=30,
                                      period_min=30, period_max=1800):
    # ACF 局部周期
    t_acf, p_acf = rolling_acf_period(edge_counts, steps,
                                      win_sec=acf_win, hop_sec=acf_hop,
                                      period_min=period_min, period_max=period_max)

    # STFT 周期能量图
    T, PER, Sxx = stft_spectrogram(edge_counts, steps,
                                   win_sec=acf_win, hop_sec=acf_hop,
                                   period_min=period_min, period_max=period_max)

    # 峰间间隔（给出总体分布的感觉）
    peaks, intervals = peak_intervals(edge_counts, min_distance=max(10, period_min//2))

    rc = {
        "font.family": "Times New Roman",
        "font.size": 13,
        "axes.labelsize": 16,
        "axes.titlesize": 16,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.linewidth": 1.2,
    }
    with mpl.rc_context(rc):
        fig = plt.figure(figsize=(12, 8))
        gs = fig.add_gridspec(nrows=3, ncols=1, height_ratios=[1.2, 1.5, 1.0], hspace=0.10)

        # 1) 顶：原始每秒切换 + ACF 局部周期（叠加在右轴）
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.plot(steps, edge_counts, color="tab:gray", lw=1.2, alpha=0.8, label="Edges/sec")
        ax1.set_ylabel("Edges/sec")
        ax1.grid(True, linestyle="--", alpha=0.3)

        ax1b = ax1.twinx()
        ax1b.plot(t_acf, p_acf, color="tab:blue", lw=2.0, label="Local period (ACF)")
        ax1b.set_ylabel("Local period (s)", color="tab:blue")
        ax1b.tick_params(axis='y', colors="tab:blue")

        # 合并图例（右侧外放）
        lines = [ax1.get_lines()[0], ax1b.get_lines()[0]]
        labels = [l.get_label() for l in lines]
        plt.tight_layout(rect=[0, 0, 0.82, 1.0])
        leg = fig.legend(lines, labels, loc="center left", bbox_to_anchor=(0.86, 0.78),
                         frameon=True, framealpha=1.0)
        leg.get_frame().set_edgecolor("black")

        # 2) 中：STFT 周期-时间能量图
        ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
        if Sxx.size:
            # 对数色标更显著
            im = ax2.pcolormesh(T, PER, (Sxx.T + 1e-12), shading="auto", cmap="viridis")
            ax2.set_yscale("log")
            ax2.set_ylabel("Period (s) [log]")
            cb = fig.colorbar(im, ax=ax2, pad=0.01)
            cb.set_label("Power")
        ax2.grid(True, linestyle=":", alpha=0.25)

        # 3) 底：峰间间隔分布 & 滚动中位数（可选）
        ax3 = fig.add_subplot(gs[2, 0], sharex=ax1)
        ax3.plot(peaks[:-1], intervals, ".", ms=3, color="tab:orange", alpha=0.7,
                 label="Inter-peak intervals")
        # 再来一个滑动中位数（窗口=10个峰）
        if len(intervals) >= 10:
            k = 10
            med = np.convolve(intervals, np.ones(k)/k, mode="valid")
            centers = peaks[1 + k//2 : 1 + k//2 + len(med)]
            ax3.plot(centers, med, color="tab:red", lw=2.0, label="Rolling median")
        ax3.set_xlabel("Time (s)")
        ax3.set_ylabel("Peak intervals (s)")
        ax3.grid(True, linestyle="--", alpha=0.3)
        ax3.legend(loc="upper right")

        plt.show()

# ===== 用法示例 =====
steps, edge_counts = compute_edge_counts_from_pending(pending_edges, fill_missing=True)
plot_switching_period_diagnostics(steps, edge_counts,
                                  acf_win=600, acf_hop=30,
                                  period_min=30, period_max=1800)


接下来，我们需要进行已有链路和正在建联链路之间的同途展示，这回让我们对整个网络的

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import csv
from typing import Mapping, Sequence, Iterable, Tuple, Dict, Any, List, Union, Set

TimeKey = int
Node = int
Snapshot = Mapping[Node, Any]
TimeSeries = Union[Mapping[TimeKey, Snapshot], Sequence[Snapshot]]

def _iter_times_and_snapshots(ts: TimeSeries, start_time: int | None) -> Iterable[Tuple[int, Snapshot]]:
    if isinstance(ts, Mapping):
        for t in sorted(ts.keys()): yield int(t), ts[t]
    elif isinstance(ts, Sequence):
        if start_time is None: start_time = 0
        for i, snap in enumerate(ts): yield start_time + i, snap
    else:
        raise TypeError("Unsupported container for time series.")

def _neighbors_from(obj: Any):
    if obj is None: return []
    if isinstance(obj, Mapping): return obj.keys()
    if isinstance(obj, (set, list, tuple)): return obj
    if isinstance(obj, (int, np.integer)): return [int(obj)]
    try: return list(obj)
    except Exception: return []

def count_unique_edges(snapshot: Snapshot, undirected: bool = True) -> int:
    if not isinstance(snapshot, Mapping):
        raise TypeError("Snapshot must be a mapping {basicSa: neighbors}.")
    seen: Set[Tuple[Node, Node]] = set()
    c = 0
    for u, nbrs in snapshot.items():
        u = int(u)
        for v in _neighbors_from(nbrs):
            v = int(v)
            if u == v: continue
            if undirected:
                a, b = (u, v) if u < v else (v, u)
                if (a, b) not in seen:
                    seen.add((a, b)); c += 1
            else:
                c += 1
    return c

def series_edge_counts(ts: TimeSeries, start_time: int | None = None, undirected: bool = True) -> Dict[int, int]:
    out: Dict[int, int] = {}
    for t, snap in _iter_times_and_snapshots(ts, start_time):
        out[int(t)] = count_unique_edges(snap, undirected=undirected)
    return out

def moving_average(y, w: int | None):
    if not w or w <= 1: return np.asarray(y, dtype=float)
    y = np.asarray(y, dtype=float)
    pad = w // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(w) / w
    return np.convolve(ypad, ker, mode="valid")

def plot_all_vs_pending(
    all_edges: TimeSeries,
    pending_edges: TimeSeries,
    start_time_all: int | None = None,
    start_time_pending: int | None = None,
    undirected: bool = True,
    smooth_window: int | None = None,
    csv_path: str | None = None,
    color_all: str = "tab:blue",
    color_pending: str = "tab:red"
):
    # 1) counts
    c_all = series_edge_counts(all_edges,  start_time=start_time_all,     undirected=undirected)
    c_pen = series_edge_counts(pending_edges, start_time=start_time_pending, undirected=undirected)

    # 2) align timeline
    times = sorted(set(c_all) | set(c_pen))
    y_all = np.array([c_all.get(t, 0) for t in times], dtype=float)
    y_pen = np.array([c_pen.get(t, 0) for t in times], dtype=float)
    y_all_s = moving_average(y_all, smooth_window)
    y_pen_s = moving_average(y_pen, smooth_window)

    # 3) plot with twin axes, distinct colors
    fig, axL = plt.subplots(figsize=(12, 5))
    axR = axL.twinx()

    ln1 = axL.plot(times, y_all_s, color=color_all, linewidth=1.8, label="all_edges (built)")
    ln2 = axR.plot(times, y_pen_s, color=color_pending, linewidth=1.8, linestyle="--", label="pending_edges (building)")

    axL.set_title("All vs Pending Edges over Time (dual y-axes)")
    axL.set_xlabel("time")
    axL.set_ylabel("all_edges (unique edges)", color=color_all)
    axR.set_ylabel("pending_edges (unique edges)", color=color_pending)

    # match tick/spine colors for readability
    axL.tick_params(axis="y", colors=color_all)
    axR.tick_params(axis="y", colors=color_pending)
    axL.spines["left"].set_color(color_all)
    axR.spines["right"].set_color(color_pending)

    axL.grid(True, linestyle="--", alpha=0.35)

    lines = ln1 + ln2
    labels = [l.get_label() for l in lines]
    axL.legend(lines, labels, loc="best")

    fig.tight_layout()
    plt.show()

    if csv_path:
        import csv
        with open(csv_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["t", "all_edges", "pending_edges"])
            for t, a, p in zip(times, y_all, y_pen):
                w.writerow([t, int(a), int(p)])
# 字典时间序列：{t: {basicSa: {dst1, dst2, ...}}, ...}

plot_all_vs_pending(all_inter_edge, pending_edges, smooth_window=5,
                    color_all="tab:blue", color_pending="tab:orange")
# 如果你的时间序列是 list/tuple，从 t=1 开始：
# plot_all_vs_pending(all_edges_list, pending_edges_list, start_time_all=1, start_time_pending=1,
#                     smooth_window=5, color_all="#2ca02c", color_pending="#d62728")


这个主要是为了计算周期，就是我要看看，这个变化是随着时间如何变化的

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from typing import Mapping, Sequence, Iterable, Tuple, Dict, Any, Union, Set, List

TimeKey = int
Node = int
Snapshot = Mapping[Node, Any]
TimeSeries = Union[Mapping[TimeKey, Snapshot], Sequence[Snapshot]]

# ---------- Helpers (与之前一致，简版复用) ----------
def _iter_times_and_snapshots(ts: TimeSeries, start_time: int | None) -> Iterable[Tuple[int, Snapshot]]:
    if isinstance(ts, Mapping):
        for t in sorted(ts.keys()): yield int(t), ts[t]
    elif isinstance(ts, Sequence):
        if start_time is None: start_time = 0
        for i, snap in enumerate(ts): yield start_time + i, snap
    else:
        raise TypeError("Unsupported container for time series.")

def _neighbors_from(obj: Any):
    if obj is None: return []
    if isinstance(obj, Mapping): return obj.keys()
    if isinstance(obj, (set, list, tuple)): return obj
    if isinstance(obj, (int, np.integer)): return [int(obj)]
    try: return list(obj)
    except Exception: return []

def count_unique_edges(snapshot: Snapshot, undirected: bool = True) -> int:
    if not isinstance(snapshot, Mapping):
        raise TypeError("Snapshot must be a mapping {basicSa: neighbors}.")
    seen: Set[Tuple[Node, Node]] = set()
    c = 0
    for u, nbrs in snapshot.items():
        u = int(u)
        for v in _neighbors_from(nbrs):
            v = int(v)
            if u == v: continue
            if undirected:
                a, b = (u, v) if u < v else (v, u)
                if (a, b) not in seen:
                    seen.add((a, b)); c += 1
            else:
                c += 1
    return c

def series_edge_counts(ts: TimeSeries, start_time: int | None = None, undirected: bool = True) -> Dict[int, int]:
    out: Dict[int, int] = {}
    for t, snap in _iter_times_and_snapshots(ts, start_time):
        out[int(t)] = count_unique_edges(snap, undirected=undirected)
    return out

def moving_average(y, w: int | None):
    if not w or w <= 1: return np.asarray(y, dtype=float)
    y = np.asarray(y, dtype=float)
    pad = w // 2
    ypad = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(w) / w
    return np.convolve(ypad, ker, mode="valid")

# ---------- New: ratio plot ----------
def plot_pending_over_all_ratio(
    all_edges: TimeSeries,
    pending_edges: TimeSeries,
    start_time_all: int | None = None,
    start_time_pending: int | None = None,
    undirected: bool = True,
    smooth_window: int | None = None,   # e.g., 5/11
    invert: bool = False,               # False: pending/all；True: all/pending
    zero_handling: str = "nan",         # "nan" | "eps" | "drop"
    eps: float = 1e-9,                  # used when zero_handling=="eps"
    csv_path: str | None = None
):
    """
    Compute and plot ratio over time.
    - invert=False: ratio = pending / all
      invert=True : ratio = all / pending
    - zero_handling:
        "nan"  : when denominator==0, ratio=NaN (kept, but not drawn)
        "eps"  : denominator += eps (avoids NaN/inf)
        "drop" : drop those timestamps entirely
    """
    c_all = series_edge_counts(all_edges,  start_time=start_time_all,     undirected=undirected)
    c_pen = series_edge_counts(pending_edges, start_time=start_time_pending, undirected=undirected)

    # align timeline (union), fill missing with 0
    times = sorted(set(c_all) | set(c_pen))
    a = np.array([c_all.get(t, 0) for t in times], dtype=float)
    p = np.array([c_pen.get(t, 0) for t in times], dtype=float)

    num, den = (p, a) if not invert else (a, p)
    if zero_handling == "eps":
        den = den + eps
        ratio = num / den
    elif zero_handling == "drop":
        mask = den != 0
        times = [times[i] for i, m in enumerate(mask) if m]
        num = num[mask]; den = den[mask]
        ratio = num / den
    else:  # "nan"
        ratio = np.divide(num, den, out=np.full_like(num, np.nan), where=den!=0)

    ratio_s = moving_average(ratio, smooth_window)

    # plot
    fig, ax = plt.subplots(figsize=(12, 4.6))
    ax.plot(times, ratio_s, linewidth=1.8)
    ax.set_title("Ratio over time: pending/all" if not invert else "Ratio over time: all/pending")
    ax.set_xlabel("time")
    ax.set_ylabel("ratio")
    ax.grid(True, linestyle="--", alpha=0.35)
    # reference lines
    ax.axhline(1.0, linestyle="--", linewidth=1.0)
    ax.axhline(0.5, linestyle=":",  linewidth=1.0)
    fig.tight_layout()
    plt.show()

    # optional CSV
    if csv_path:
        import csv
        with open(csv_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["t", "all_edges", "pending_edges", "ratio"])
            for t, aa, pp, rr in zip(times, a, p, ratio):
                # if dropped we already removed; if NaN keep NaN
                w.writerow([t, int(aa) if not np.isnan(aa) else "",
                               int(pp) if not np.isnan(pp) else "",
                               rr if not (isinstance(rr, float) and np.isnan(rr)) else ""])

# ---------------------------
# 用法示例：
# 1) 默认：比率 = pending / all，平滑窗=5，分母为0时记 NaN（图上自动断开）
plot_pending_over_all_ratio(all_edges, pending_edges, smooth_window=5, zero_handling="nan")
#
# 2) 若你的时间序列是 list/tuple 且从 t=1 开始：
# plot_pending_over_all_ratio(all_edges_list, pending_edges_list, start_time_all=1, start_time_pending=1,
#                             smooth_window=5, zero_handling="eps", eps=1e-6)
#
# 3) 想看 all/pending：
# plot_pending_over_all_ratio(all_edges, pending_edges, invert=True, smooth_window=11)
#
# 4) 导出 CSV：
# plot_pending_over_all_ratio(all_edges, pending_edges, smooth_window=5, csv_path="ratio_series.csv")
# ---------------------------
